 
# QUALIFYING IMPORTANCE ACROSS F1 / INDYCAR / NASCAR

 ## Notebook Summary
---------------
Models

1. Finish ~ Qualifying + Race
2. Finish ~ DriverQuality + Race
3. Finish ~ Qualifying + DriverQuality + Race

4. Finish ~ TeamQuality + Race
5. Finish ~ Qualifying + TeamQuality + Race

6. Finish ~ Qualifying + DriverQuality + TeamQuality + Race

Main statistic:

incremental R^2 from qualifying after accounting for quality.




## Read Data

In [60]:

#============================================================
# 0. PACKAGES
#============================================================
from pandas._libs.writers import write_csv_rows
import glob
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")


# ============================================================
# 1. PATHS
# ============================================================

DATA_DIR = "/Users/archith/Downloads/racing-main"

OUTPUT_DIR = os.path.join(DATA_DIR, "qualifying_importance_analysis")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. FIND CSV FILES
# ============================================================

all_files = glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)

csv_files = [f for f in all_files if re.search(r"\.csv$", f, re.IGNORECASE)]


# ============================================================
# 3. IDENTIFY SERIES FROM FILE PATH
# ============================================================

def get_series(path: str):
    p = path.lower()

    if re.search(r"indycar", p):
        return "IndyCar"
    if re.search(r"nascar", p):
        return "NASCAR"
    if re.search(r"f1|formula.?1", p):
        return "F1"
    return None


# ============================================================
# 4. IDENTIFY YEAR
# ============================================================

def get_year(path: str):
    matches = re.findall(r"(?<!\d)20\d{2}(?!\d)", path)

    if len(matches) == 0:
        return np.nan

    return int(matches[0])


# ============================================================
# 5. FIND COLUMN
# ============================================================

def find_column(df: pd.DataFrame, candidates):
    lower_names = {c.lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in lower_names:
            return lower_names[cand.lower()]

    return None


# ============================================================
# 6. CONVERT POSITION TO NUMERIC
# ============================================================

def clean_position(x: pd.Series):
    if pd.api.types.is_numeric_dtype(x):
        return pd.to_numeric(x, errors="coerce")

    x = x.astype(str)
    extracted = x.str.extract(r"(\d+)")[0]

    return pd.to_numeric(extracted, errors="coerce")

def extract_race_num_id(text: str):
    m = re.match(r"(\d+)", str(text))
    return m.group(1) if m else None


## Join qualifying position to data

In [61]:

# ============================================================
# 7. READ LAP FILE
# ============================================================

def read_lap_file(path: str):
    series = get_series(path)

    if series is None:
        return None

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        print("FAILED:", os.path.basename(path))
        print("ERROR:", repr(e))
        return None
        

    # ----------------------------------------------------------
    # COLUMN IDENTIFICATION
    # ----------------------------------------------------------

    lap_col = find_column(df, ["lap_number", "lap", "lap_num", "lap_no"])

    driver_col = find_column(
        df, ["driver_id", "driver", "driver_name", "driver_number", "name"]
    )

    team_col = find_column(
        df,
        ["team", "team_name", "constructor", "constructor_name", "car", "manufacturer"],
    )

    position_col = find_column(
        df, ["running_position", "position", "race_position", "track_position"]
    )

    # ----------------------------------------------------------
    # QUALIFYING / GRID POSITION
    # ----------------------------------------------------------

    qualifying_col = find_column(
        df,
        [
            "qualifying_position",
            "qualifying_pos",
            "qual_pos",
            "grid_position",
            "grid_pos",
            "starting_position",
            "start_position",
            "start_pos",
            "grid",
        ],
    )

    # ----------------------------------------------------------
    # FINISHING POSITION
    # ----------------------------------------------------------

    finish_col = find_column(
        df,
        [
            "finish_position",
            "finishing_position",
            "final_position",
            "result_position",
            "race_result",
        ],
    )

    # ----------------------------------------------------------
    # LAP TIME
    # ----------------------------------------------------------

    if series == "F1":
        lap_time_col = find_column(df, ["lap_time_ms"])
    else:
        lap_time_col = find_column(df, ["lap_time"])

    # ----------------------------------------------------------
    # PIT INFORMATION
    # ----------------------------------------------------------

    pit_col = find_column(
        df, ["pit", "pit_stop", "pit_stop_flag", "pitted", "is_pit_lap", "pit_lap"]
    )

    # ----------------------------------------------------------
    # REQUIRED COLUMNS
    # ----------------------------------------------------------

    required = [lap_col, driver_col, position_col, lap_time_col]

    if any(c is None for c in required):
        print("SKIPPED:", os.path.basename(path))
        print("Missing required column.")
        return None

    # ----------------------------------------------------------
    # RACE IDENTIFIER
    # ----------------------------------------------------------

    race_file = Path(path).stem

    # ----------------------------------------------------------
    # STANDARDIZED DATA
    # ----------------------------------------------------------

    result = pd.DataFrame(
        {
            "series": series,
            "year": get_year(path),
            "race_file": race_file,
            "driver_key": df[driver_col].astype(str),
            "team_key": df[team_col].astype(str) if team_col is not None else np.nan,
            "lap": pd.to_numeric(df[lap_col], errors="coerce"),
            "position": clean_position(df[position_col]),
        }
    )

    # ----------------------------------------------------------
    # QUALIFYING POSITION
    # ----------------------------------------------------------

    if qualifying_col is not None:
        result["qualifying_position"] = clean_position(df[qualifying_col])
    else:
        result["qualifying_position"] = np.nan

    # ----------------------------------------------------------
    # FINISHING POSITION
    # ----------------------------------------------------------

    if finish_col is not None:
        result["finish_position"] = clean_position(df[finish_col])
    else:
        result["finish_position"] = np.nan

    # ----------------------------------------------------------
    # LAP TIME
    # ----------------------------------------------------------

    result["lap_time_raw"] = pd.to_numeric(df[lap_time_col], errors="coerce")

    if series == "F1":
        result["lap_time_sec"] = result["lap_time_raw"] / 1000
    else:
        result["lap_time_sec"] = result["lap_time_raw"]

    # ----------------------------------------------------------
    # PIT FLAG
    # ----------------------------------------------------------

    if pit_col is not None:
        pit_value = df[pit_col]

        if pd.api.types.is_bool_dtype(pit_value):
            result["pit_lap"] = pit_value
        elif pd.api.types.is_numeric_dtype(pit_value):
            result["pit_lap"] = pit_value != 0
        else:
            result["pit_lap"] = (
                pit_value.astype(str)
                .str.lower()
                .isin(["true", "yes", "y", "1", "pit", "pitted"])
            )
    else:
        result["pit_lap"] = np.nan

    return result

drivers_files = [f for f in csv_files if re.search(r"(^|[\\/])drivers\.csv$", f, re.IGNORECASE)]

DRIVER_ID_CANDIDATES = ["driver_id", "driver_number", "code", "driver", "driver_name", "name"]

def load_drivers_file(path: str):
    series = get_series(path)
    if series is None:
        print("SKIPPED drivers.csv (could not detect series):", path)
        return None

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        print("FAILED (drivers.csv):", path)
        print("ERROR:", repr(e))
        return None

    race_id_col = find_column(df, ["race_id"])
    start_col = find_column(df, ["start"])
    finish_col = find_column(df, ["finish"])

    if any(c is None for c in [race_id_col, start_col, finish_col]):
        print(f"SKIPPED drivers.csv for {series} (missing race_id/start/finish):", path)
        return None

    out = pd.DataFrame(
        {
            "series": series,
            "race_file": df[race_id_col].astype(str),
            "drivers_csv_start": clean_position(df[start_col]),
            "drivers_csv_finish": clean_position(df[finish_col]),
        }
    )

    # Keep every plausible identifier column found, prefixed so we can test them later
    found_any = False
    for cand in DRIVER_ID_CANDIDATES:
        col = find_column(df, [cand])
        if col is not None:
            out[f"cand__{cand}"] = df[col].astype(str)
            found_any = True

    if not found_any:
        print(f"SKIPPED drivers.csv for {series} (no driver identifier column found):", path)
        return None

    return out

drivers_list = [load_drivers_file(f) for f in drivers_files]
drivers_list = [d for d in drivers_list if d is not None]

drivers_master = (
    pd.concat(drivers_list, ignore_index=True) if drivers_list else pd.DataFrame()
)

print("drivers.csv rows loaded:", len(drivers_master))
if len(drivers_master) > 0:
    print(drivers_master.groupby("series").size())

# ============================================================
# 8. READ LAP FILES
# ============================================================

lap_files = [f for f in csv_files if re.search(r"lap", f, re.IGNORECASE)]
lap_files = [f for f in lap_files if get_series(f) is not None]

print("Lap files found:", len(lap_files), "\n")

lap_data_list = [read_lap_file(f) for f in lap_files]
lap_data_list = [d for d in lap_data_list if d is not None]

lap_data = (
    pd.concat(lap_data_list, ignore_index=True)
    if lap_data_list
    else pd.DataFrame()
)


# ============================================================
# 9. BASIC CLEANING
# ============================================================

lap_data = lap_data[
    lap_data["series"].notna()
    & lap_data["year"].notna()
    & lap_data["driver_key"].notna()
    & lap_data["lap"].notna()
    & lap_data["position"].notna()
    & lap_data["lap_time_sec"].notna()
    & (lap_data["lap_time_sec"] > 0)
    & np.isfinite(lap_data["lap_time_sec"])
].copy()

lap_data["race_id"] = (
    lap_data["series"].astype(str)
    + "_"
    + lap_data["year"].astype(int).astype(str)
    + "_"
    + lap_data["race_file"].astype(str)
)

lap_data["driver_key"] = lap_data["driver_key"].astype(str)
lap_data["team_key"] = lap_data["team_key"].fillna("Unknown").astype(str)

# ============================================================
# 9b. PICK BEST driver_key COLUMN FOR drivers.csv, PER SERIES
#
# drivers.csv may identify drivers differently per series
# (e.g. driver_id for F1/NASCAR, driver_name for IndyCar).
# Rather than assume, pick whichever candidate column
# overlaps best with the driver_key values already used
# in the lap data for that series.
# ============================================================

if len(drivers_master) > 0:
    resolved_rows = []

    for series_name, sub in drivers_master.groupby("series"):
        lap_keys = set(lap_data.loc[lap_data["series"] == series_name, "driver_key"])

        cand_cols = [c for c in sub.columns if c.startswith("cand__")]
        best_col, best_overlap = None, -1

        for c in cand_cols:
            overlap = sub[c].isin(lap_keys).mean()
            print(f"{series_name}: {c} overlap = {overlap:.2%}")
            if overlap > best_overlap:
                best_overlap = overlap
                best_col = c

        print(f"--> {series_name}: using {best_col} (overlap {best_overlap:.2%})\n")

        sub = sub.copy()
        sub["driver_key"] = sub[best_col]
        resolved_rows.append(
            sub[["series", "race_file", "driver_key", "drivers_csv_start", "drivers_csv_finish"]]
        )

    drivers_master = pd.concat(resolved_rows, ignore_index=True)

# ============================================================
# 10. REMOVE OBVIOUSLY INVALID LAP TIMES
#
# We don't want timing errors dominating quality.
#
# Within each race/lap, remove times that are extreme
# relative to the field.
#
# ============================================================

def mad_r(s: pd.Series):
    """R-style mad(): 1.4826 * median(|x - median(x)|), NAs dropped."""
    s = s.dropna()
    if len(s) == 0:
        return np.nan
    med = s.median()
    return 1.4826 * (s - med).abs().median()


lap_data["lap_median"] = lap_data.groupby(["race_id", "lap"])["lap_time_sec"].transform(
    "median"
)
lap_data["lap_mad"] = lap_data.groupby(["race_id", "lap"])["lap_time_sec"].transform(
    mad_r
)

lap_data = lap_data[
    lap_data["lap_mad"].isna()
    | (lap_data["lap_mad"] == 0)
    | ((lap_data["lap_time_sec"] - lap_data["lap_median"]).abs() < 10 * lap_data["lap_mad"])
].copy()


# ============================================================
# 11. EXCLUDE PIT LAPS WHEN WE KNOW THEY ARE PIT LAPS
#
# F1:
#     no pit information -> don't exclude anything.
#
# IndyCar/NASCAR:
#     known pit laps excluded.
#
# ============================================================

pace_data = lap_data[
    lap_data["pit_lap"].isna() | (lap_data["pit_lap"] == False)  # noqa: E712
].copy()


drivers.csv rows loaded: 15628
series
F1         2860
IndyCar    3040
NASCAR     9728
dtype: int64
Lap files found: 529 

F1: cand__driver_id overlap = 100.00%
F1: cand__driver overlap = 0.00%
--> F1: using cand__driver_id (overlap 100.00%)

IndyCar: cand__driver_id overlap = 0.00%
IndyCar: cand__driver overlap = 100.00%
--> IndyCar: using cand__driver (overlap 100.00%)

NASCAR: cand__driver_id overlap = 100.00%
NASCAR: cand__driver overlap = 0.00%
--> NASCAR: using cand__driver_id (overlap 100.00%)



## Adjustment by quality

In [62]:
# ============================================================
# 12. WITHIN-RACE/LAP NORMALIZATION
# ============================================================
#
# Raw lap times cannot be compared across tracks.
#
# Instead:
#
#   relative_lap_time =
#       driver's lap time
#       - field median
#
# Negative = faster than field.
#
# We also create a z-score.
#
# ============================================================

grp = pace_data.groupby(["race_id", "lap"])["lap_time_sec"]

pace_data["field_median"] = grp.transform("median")
pace_data["field_sd"] = grp.transform("std")

pace_data["relative_lap_time"] = pace_data["lap_time_sec"] - pace_data["field_median"]

pace_data["relative_pace_z"] = np.where(
    pace_data["field_sd"] > 0,
    pace_data["relative_lap_time"] / pace_data["field_sd"],
    np.nan,
)


# ============================================================
# 13. DRIVER x RACE PACE
# ============================================================

driver_race_pace = (
    pace_data.groupby(
        ["race_id", "series", "year", "race_file", "driver_key"], as_index=False
    )
    .agg(
        race_laps=("relative_pace_z", "size"),
        race_relative_pace=("relative_pace_z", "median"),
        race_relative_time=("relative_lap_time", "median"),
    )
)


# ============================================================
# 14. TEAM x RACE PACE
# ============================================================

team_race_pace = (
    pace_data.groupby(
        ["race_id", "series", "year", "race_file", "team_key"], as_index=False
    )
    .agg(
        team_race_relative_pace=("relative_pace_z", "median"),
        team_race_relative_time=("relative_lap_time", "median"),
    )
)


# ============================================================
# 15. DRIVER QUALITY
#
# IMPORTANT:
#
# For a driver in race R, their quality is calculated using
# OTHER races only (leave-one-race-out average).
#
# ============================================================

driver_quality_base = (
    driver_race_pace.groupby(["series", "driver_key"], as_index=False).agg(
        quality_sum=("race_relative_pace", lambda s: s.sum(skipna=True)),
        quality_n=("race_relative_pace", lambda s: s.notna().sum()),
    )
)

driver_quality = driver_race_pace[
    ["race_id", "series", "driver_key", "race_relative_pace"]
].merge(driver_quality_base, on=["series", "driver_key"], how="left")

driver_quality["driver_quality"] = np.where(
    driver_quality["quality_n"] > 1,
    (driver_quality["quality_sum"] - driver_quality["race_relative_pace"])
    / (driver_quality["quality_n"] - 1),
    np.nan,
)

driver_quality = driver_quality[["race_id", "series", "driver_key", "driver_quality"]]


# ============================================================
# 16. TEAM QUALITY
#
# Same leave-one-race-out approach.
# ============================================================

team_quality_base = (
    team_race_pace.groupby(["series", "team_key"], as_index=False).agg(
        quality_sum=("team_race_relative_pace", lambda s: s.sum(skipna=True)),
        quality_n=("team_race_relative_pace", lambda s: s.notna().sum()),
    )
)

team_quality = team_race_pace[
    ["race_id", "series", "team_key", "team_race_relative_pace"]
].merge(team_quality_base, on=["series", "team_key"], how="left")

team_quality["team_quality"] = np.where(
    team_quality["quality_n"] > 1,
    (team_quality["quality_sum"] - team_quality["team_race_relative_pace"])
    / (team_quality["quality_n"] - 1),
    np.nan,
)

team_quality = team_quality[["race_id", "series", "team_key", "team_quality"]]


# ============================================================
# 17. CREATE DRIVER x RACE RESULT DATA
# ============================================================

def first_or_nan(s: pd.Series):
    s2 = s.dropna()
    return s2.iloc[0] if len(s2) > 0 else np.nan


def summarize_race_group(g: pd.DataFrame):
    g = g.sort_values("lap")

    # Qualifying position
    q = g["qualifying_position"].dropna()
    qualifying_position = q.iloc[0] if len(q) > 0 else np.nan

    # Finishing position
    f = g["finish_position"].dropna()
    if len(f) > 0:
        finish_position = f.iloc[0]
    else:
        finish_position = g.loc[g["lap"].idxmax(), "position"]

    # Position on Lap 1
    lap1 = g.loc[g["lap"] == 2, "position"].dropna()
    lap1_position = lap1.iloc[0] if len(lap1) > 0 else np.nan

    # Keep your existing fallback/start position as well
    starting_observed_position = g.loc[g["lap"].idxmin(), "position"]

    laps_completed = g["lap"].max()

    return pd.Series(
        {
            "qualifying_position": qualifying_position,
            "lap1_position": lap1_position,
            "finish_position": finish_position,
            "starting_observed_position": starting_observed_position,
            "laps_completed": laps_completed,
        }
    )


lap_data_sorted = lap_data.sort_values(["race_id", "driver_key", "lap"])

race_results = (
    lap_data_sorted.groupby(
        ["race_id", "series", "year", "race_file", "driver_key", "team_key"]
    )
    .apply(summarize_race_group)
    .reset_index()
)
# ============================================================
# 17b. OVERLAY AUTHORITATIVE QUALIFYING/FINISH FROM drivers.csv
# ============================================================

if len(drivers_master) > 0:
    race_results["race_num_id"] = race_results["race_file"].apply(extract_race_num_id)
    drivers_master["race_num_id"] = drivers_master["race_file"].apply(extract_race_num_id)

    race_results = race_results.merge(
        drivers_master.drop(columns=["race_file"]),
        on=["series", "race_num_id", "driver_key"],
        how="left",
    )

    race_results["qualifying_position"] = race_results["drivers_csv_start"].combine_first(
        race_results["qualifying_position"]
    )
    race_results["finish_position"] = race_results["drivers_csv_finish"].combine_first(
        race_results["finish_position"]
    )

    n_matched = race_results["drivers_csv_start"].notna().sum()
    print(f"Matched {n_matched} / {len(race_results)} driver-race rows to drivers.csv")

    race_results = race_results.drop(
        columns=["drivers_csv_start", "drivers_csv_finish", "race_num_id"]
    )

# ============================================================
# 18. FALLBACK QUALIFYING POSITION
#
# If no explicit qualifying/grid column exists, use the
# first observed race position.
#
# IMPORTANT:
# This is only a fallback.
#
# ============================================================

n_missing_q = race_results["qualifying_position"].isna().sum()

if n_missing_q > 0:
    print(
        f"\nWARNING: {n_missing_q} driver-race observations have no explicit "
        "qualifying position."
    )
    print("Using first observed race position as fallback.\n")

    race_results["qualifying_position"] = race_results["qualifying_position"].fillna(
        race_results["starting_observed_position"]
    )


# ============================================================
# 19. POSITION GAIN
# ============================================================

race_results["position_gain"] = (
    race_results["qualifying_position"] - race_results["finish_position"]
)
race_results["absolute_position_change"] = race_results["position_gain"].abs()



Matched 15413 / 15413 driver-race rows to drivers.csv


## Add quality adjustment to data + helpers for model

In [63]:

# ============================================================
# 20. JOIN QUALITY
# ============================================================

analysis_data = race_results.merge(
    driver_quality, on=["race_id", "series", "driver_key"], how="left"
).merge(team_quality, on=["race_id", "series", "team_key"], how="left")


# ============================================================
# 21. QUALITY DIRECTION
#
# Negative pace = faster.
#
# We reverse the sign so:
#
# HIGHER QUALITY = BETTER.
#
# ============================================================

analysis_data["driver_quality"] = -analysis_data["driver_quality"]
analysis_data["team_quality"] = -analysis_data["team_quality"]


# ============================================================
# 22. STANDARDIZE QUALITY WITHIN SERIES
#
# Makes driver/team coefficients comparable.
# (R's scale() uses the sample standard deviation, ddof=1.)
# ============================================================

def zscore_ddof1(s: pd.Series):
    return (s - s.mean()) / s.std(ddof=1)


analysis_data["driver_quality_z"] = analysis_data.groupby("series")[
    "driver_quality"
].transform(zscore_ddof1)

analysis_data["team_quality_z"] = analysis_data.groupby("series")[
    "team_quality"
].transform(zscore_ddof1)


# ============================================================
# 23. CLEAN MODEL DATA
# ============================================================

analysis_data = analysis_data[
    analysis_data["qualifying_position"].notna()
    & analysis_data["finish_position"].notna()
].copy()


# ============================================================
# 24. ROBUST MODEL FUNCTION
#
# Race fixed effects are included when there are at least
# two races available in the relevant model sample.
#
# This prevents the model from failing when quality filtering
# leaves only one race in a series.
# ============================================================

def fit_model(formula_main: str, df: pd.DataFrame):
    if len(df) < 10:
        return None

    n_races = df["race_id"].nunique()

    # --------------------------------------------------------
    # If there are multiple races: include race fixed effects
    # --------------------------------------------------------

    if n_races >= 2:
        formula_full = f"{formula_main} + C(race_id)"

        try:
            return smf.ols(formula_full, data=df).fit()
        except Exception:
            return None

    # --------------------------------------------------------
    # Only one race: cannot estimate race fixed effects
    # meaningfully -> return None (mirrors the R behavior).
    # --------------------------------------------------------

    return None


# ============================================================
# 25. MODEL FUNCTION
# ============================================================

def get_r2(model):
    if model is None:
        return np.nan
    return model.rsquared


def run_models(df: pd.DataFrame):
    # ----------------------------------------------------------
    # DRIVER MODEL DATA
    # ----------------------------------------------------------

    df_driver = df[
        df["driver_quality_z"].notna()
        & df["qualifying_position"].notna()
        & df["finish_position"].notna()
    ].copy()

    # ----------------------------------------------------------
    # TEAM MODEL DATA
    # ----------------------------------------------------------

    df_team = df[
        df["team_quality_z"].notna()
        & df["qualifying_position"].notna()
        & df["finish_position"].notna()
    ].copy()

    # ----------------------------------------------------------
    # FULL MODEL DATA
    # ----------------------------------------------------------

    df_full = df[
        df["driver_quality_z"].notna()
        & df["team_quality_z"].notna()
        & df["qualifying_position"].notna()
        & df["finish_position"].notna()
    ].copy()

    # ----------------------------------------------------------
    # FIT MODELS
    # ----------------------------------------------------------

    m_q_driver = fit_model("finish_position ~ qualifying_position", df_driver)
    m_d = fit_model("finish_position ~ driver_quality_z", df_driver)
    m_qd = fit_model(
        "finish_position ~ qualifying_position + driver_quality_z", df_driver
    )

    m_q_team = fit_model("finish_position ~ qualifying_position", df_team)
    m_t = fit_model("finish_position ~ team_quality_z", df_team)
    m_qt = fit_model(
        "finish_position ~ qualifying_position + team_quality_z", df_team
    )

    m_full = fit_model(
        "finish_position ~ qualifying_position + driver_quality_z + team_quality_z",
        df_full,
    )

    # ----------------------------------------------------------
    # R^2
    # ----------------------------------------------------------

    r2_q_driver = get_r2(m_q_driver)
    r2_d = get_r2(m_d)
    r2_qd = get_r2(m_qd)

    r2_q_team = get_r2(m_q_team)
    r2_t = get_r2(m_t)
    r2_qt = get_r2(m_qt)

    r2_full = get_r2(m_full)

    # ----------------------------------------------------------
    # QUALIFYING INCREMENTAL R^2
    # ----------------------------------------------------------

    incremental_q_driver = r2_qd - r2_d
    incremental_q_team = r2_qt - r2_t

    m_full_no_q = fit_model(
        "finish_position ~ driver_quality_z + team_quality_z", df_full
    )
    r2_full_no_q = get_r2(m_full_no_q)
    incremental_q_full = r2_full - r2_full_no_q

    # ----------------------------------------------------------
    # PARTIAL R^2
    # ----------------------------------------------------------

    if pd.notna(r2_d) and pd.notna(r2_qd) and r2_d < 1:
        partial_q_driver = (r2_qd - r2_d) / (1 - r2_d)
    else:
        partial_q_driver = np.nan

    if pd.notna(r2_t) and pd.notna(r2_qt) and r2_t < 1:
        partial_q_team = (r2_qt - r2_t) / (1 - r2_t)
    else:
        partial_q_team = np.nan

    # ----------------------------------------------------------
    # QUALITY INCREMENTAL R^2
    # ----------------------------------------------------------

    incremental_driver_quality = r2_qd - r2_q_driver
    incremental_team_quality = r2_qt - r2_q_team

    # ----------------------------------------------------------
    # OUTPUT
    # ----------------------------------------------------------

    return pd.DataFrame(
        [
            {
                "n_driver_model": len(df_driver),
                "n_team_model": len(df_team),
                "n_full_model": len(df_full),
                "races_driver": df_driver["race_id"].nunique(),
                "races_team": df_team["race_id"].nunique(),
                "races_full": df_full["race_id"].nunique(),
                # Driver
                "r2_qualifying_driver": r2_q_driver,
                "r2_driver_quality": r2_d,
                "r2_qualifying_driver_quality": r2_qd,
                "incremental_r2_qualifying_driver": incremental_q_driver,
                "partial_r2_qualifying_driver": partial_q_driver,
                "incremental_r2_driver_quality": incremental_driver_quality,
                # Team
                "r2_qualifying_team": r2_q_team,
                "r2_team_quality": r2_t,
                "r2_qualifying_team_quality": r2_qt,
                "incremental_r2_qualifying_team": incremental_q_team,
                "partial_r2_qualifying_team": partial_q_team,
                "incremental_r2_team_quality": incremental_team_quality,
                # Full
                "r2_full": r2_full,
                "incremental_r2_qualifying_full": incremental_q_full,
            }
        ]
    )



## Run models

In [64]:

# ============================================================
# 26. RUN MODELS BY SERIES
# ============================================================

model_summary = (
    analysis_data.groupby("series", group_keys=True)
    .apply(lambda g: run_models(g))
    .reset_index(level=0)
    .reset_index(drop=True)
)

print(model_summary)


# ============================================================
# 28. SIMPLE DESCRIPTIVE SUMMARY
# ============================================================

descriptive_summary = analysis_data.groupby("series", as_index=False).agg(
    races=("race_id", "nunique"),
    driver_race_observations=("race_id", "size"),
    mean_qualifying_position=("qualifying_position", "mean"),
    mean_finish_position=("finish_position", "mean"),
    mean_position_gain=("position_gain", "mean"),
    sd_position_gain=("position_gain", "std"),
    mean_absolute_position_change=("absolute_position_change", "mean"),
)


# ============================================================
# 29. RACE-LEVEL QUALIFYING EFFECT
#
# Estimate the qualifying/finish relationship independently
# within each race.
#
# This tells us whether the pooled result is driven by a few
# races.
# ============================================================

def race_level_effect(df: pd.DataFrame):
    df = df[df["qualifying_position"].notna() & df["finish_position"].notna()]

    if len(df) < 5:
        return pd.Series(
            {
                "n": len(df),
                "qualifying_coefficient": np.nan,
                "r_squared": np.nan,
                "spearman_correlation": np.nan,
            }
        )

    model = smf.ols("finish_position ~ qualifying_position", data=df).fit()

    spearman_corr, _ = stats.spearmanr(
        df["qualifying_position"], df["finish_position"]
    )

    return pd.Series(
        {
            "n": len(df),
            "qualifying_coefficient": model.params.get("qualifying_position", np.nan),
            "r_squared": model.rsquared,
            "spearman_correlation": spearman_corr,
        }
    )


race_effects = (
    analysis_data.groupby(["series", "race_id"])
    .apply(race_level_effect)
    .reset_index()
)


# ============================================================
# 30. DISTRIBUTION OF RACE-LEVEL R^2
# ============================================================

race_r2_summary = race_effects.groupby("series", as_index=False).agg(
    median_r2=("r_squared", "median"),
    mean_r2=("r_squared", "mean"),
    q25_r2=("r_squared", lambda s: s.quantile(0.25)),
    q75_r2=("r_squared", lambda s: s.quantile(0.75)),
)

    series  n_driver_model  n_team_model  n_full_model  races_driver  \
0       F1            2778          2779          2778           142   
1  IndyCar            2908          2913          2906           113   
2   NASCAR            9695          9718          9695           272   

   races_team  races_full  r2_qualifying_driver  r2_driver_quality  \
0         142         142              0.381624           0.338750   
1         113         113              0.210086           0.218251   
2         272         272              0.197758           0.225537   

   r2_qualifying_driver_quality  ...  partial_r2_qualifying_driver  \
0                      0.451317  ...                      0.170234   
1                      0.272109  ...                      0.068894   
2                      0.257979  ...                      0.041890   

   incremental_r2_driver_quality  r2_qualifying_team  r2_team_quality  \
0                       0.069693            0.381734         0.356838   
1  

## Generate analysis plots

In [65]:
# ============================================================
# 31. PLOT: QUALIFYING VS FINISH
# ============================================================

series_list = sorted(analysis_data["series"].dropna().unique())

fig, axes = plt.subplots(1, len(series_list), figsize=(11, 7), sharey=True)
if len(series_list) == 1:
    axes = [axes]

for ax, s in zip(axes, series_list):
    sub = analysis_data[analysis_data["series"] == s]
    ax.scatter(sub["qualifying_position"], sub["finish_position"], alpha=0.20, s=15)

    if len(sub) > 1:
        coeffs = np.polyfit(sub["qualifying_position"], sub["finish_position"], 1)
        xs = np.linspace(
            sub["qualifying_position"].min(), sub["qualifying_position"].max(), 100
        )
        ax.plot(xs, np.polyval(coeffs, xs), color="tab:blue")

    ax.set_title(s)
    ax.set_xlabel("Qualifying position")

axes[0].set_ylabel("Finishing position")
fig.suptitle("Qualifying position and finishing position\nDriver-race observations")
fig.tight_layout()
fig.savefig(
    os.path.join(OUTPUT_DIR, "01_qualifying_vs_finish.png"), dpi=300
)
plt.close(fig)


# ============================================================
# 32. PLOT: POSITION GAIN DISTRIBUTION
# ============================================================

fig, axes = plt.subplots(1, len(series_list), figsize=(11, 7), sharey=True)
if len(series_list) == 1:
    axes = [axes]

for ax, s in zip(axes, series_list):
    sub = analysis_data[analysis_data["series"] == s]
    ax.hist(sub["position_gain"].dropna(), bins=30)
    ax.axvline(0, linestyle="--", color="black")
    ax.set_title(s)
    ax.set_xlabel("Positions gained (+) / lost (-)")

axes[0].set_ylabel("Drivers")
fig.suptitle("Position gained or lost from qualifying to finish")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "02_position_gain.png"), dpi=300)
plt.close(fig)


# ============================================================
# 33. PLOT: INCREMENTAL QUALIFYING R^2
# ============================================================

r2_plot_data = model_summary[
    ["series", "partial_r2_qualifying_driver", "partial_r2_qualifying_team"]
].melt(id_vars="series", var_name="adjustment", value_name="partial_r2")

r2_plot_data["adjustment"] = r2_plot_data["adjustment"].replace(
    {
        "partial_r2_qualifying_driver": "Adjusted for driver quality",
        "partial_r2_qualifying_team": "Adjusted for team quality",
    }
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=r2_plot_data, x="series", y="partial_r2", hue="adjustment", ax=ax)
ax.set_title(
    "Incremental explanatory power of qualifying\n"
    "After accounting for underlying quality and race effects"
)
ax.set_xlabel("Series")
ax.set_ylabel("Partial R\u00b2")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "03_qualifying_partial_r2.png"), dpi=300)
plt.close(fig)


# ============================================================
# 34. PLOT: QUALIFYING VS QUALITY CONTRIBUTION
# ============================================================

contribution_plot = model_summary[
    ["series", "incremental_r2_qualifying_driver", "incremental_r2_driver_quality"]
].melt(id_vars="series", var_name="variable", value_name="incremental_r2")

contribution_plot["variable"] = contribution_plot["variable"].replace(
    {
        "incremental_r2_qualifying_driver": "Qualifying",
        "incremental_r2_driver_quality": "Driver quality",
    }
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=contribution_plot, x="series", y="incremental_r2", hue="variable", ax=ax
)
ax.set_title(
    "Qualifying vs. driver quality\n"
    "Incremental explanatory power after controlling for the other variable"
)
ax.set_xlabel("Series")
ax.set_ylabel("Incremental R\u00b2")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "04_qualifying_vs_quality.png"), dpi=300)
plt.close(fig)


# ============================================================
# 35. PLOT: RACE-LEVEL QUALIFYING R^2
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))
sns.boxplot(
    data=race_effects, x="series", y="r_squared", ax=ax, showfliers=False, boxprops=dict(alpha=0.5)
)
sns.stripplot(
    data=race_effects, x="series", y="r_squared", ax=ax, alpha=0.20, jitter=0.15, color="black"
)
ax.set_title("Race-to-race variation in qualifying importance")
ax.set_xlabel("Series")
ax.set_ylabel("Race-level R\u00b2")
fig.tight_layout()
fig.savefig(
    os.path.join(OUTPUT_DIR, "05_race_level_qualifying_r2.png"), dpi=300
)
plt.close(fig)


# ============================================================
# 36. PLOT: QUALITY VS QUALIFYING
# ============================================================

fig, axes = plt.subplots(1, len(series_list), figsize=(11, 7), sharey=True)
if len(series_list) == 1:
    axes = [axes]

for ax, s in zip(axes, series_list):
    sub = analysis_data[
        (analysis_data["series"] == s) & analysis_data["driver_quality_z"].notna()
    ]
    ax.scatter(sub["driver_quality_z"], sub["qualifying_position"], alpha=0.20, s=15)

    if len(sub) > 1:
        coeffs = np.polyfit(sub["driver_quality_z"], sub["qualifying_position"], 1)
        xs = np.linspace(
            sub["driver_quality_z"].min(), sub["driver_quality_z"].max(), 100
        )
        ax.plot(xs, np.polyval(coeffs, xs), color="tab:blue")

    ax.set_title(s)
    ax.set_xlabel("Driver quality\n(higher = better)")

axes[0].set_ylabel("Qualifying position")
fig.suptitle("Underlying driver quality and qualifying position")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "06_quality_vs_qualifying.png"), dpi=300)
plt.close(fig)



## Final model

In [66]:
# ============================================================
# COMBINED DRIVER + TEAM QUALITY MODEL (illustrative, matches
# the ad-hoc `common` block in the original R script)
# ============================================================

common = analysis_data[
    analysis_data["finish_position"].notna()
    & analysis_data["qualifying_position"].notna()
    & analysis_data["driver_quality_z"].notna()
    & analysis_data["team_quality_z"].notna()
].copy()

if len(common) > 0 and common["race_id"].nunique() >= 2:
    m_quality = smf.ols(
        "finish_position ~ driver_quality_z + team_quality_z + C(race_id)",
        data=common,
    ).fit()

    m_quality_q = smf.ols(
        "finish_position ~ qualifying_position + driver_quality_z + team_quality_z + C(race_id)",
        data=common,
    ).fit()

    r2_quality = m_quality.rsquared
    r2_quality_q = m_quality_q.rsquared
    incremental_r2 = r2_quality_q - r2_quality


# ============================================================
# EXACT COMMON-SAMPLE QUALIFYING INCREMENTAL R^2
# F1 vs IndyCar vs NASCAR
#
# SAME SPECIFICATION FOR ALL THREE:
#
# Base:
#   finish ~ driver_quality + race FE
#
# Full:
#   finish ~ qualifying + driver_quality + race FE
#
# Both models use EXACTLY THE SAME OBSERVATIONS.
# ============================================================

def common_sample_r2(df: pd.DataFrame):
    if df["race_id"].nunique() < 2:
        return pd.DataFrame(
            [
                {
                    "n": len(df),
                    "races": df["race_id"].nunique(),
                    "r2_quality": np.nan,
                    "r2_quality_qualifying": np.nan,
                    "incremental_r2": np.nan,
                    "partial_r2_qualifying": np.nan,
                }
            ]
        )

    m_quality = smf.ols(
        "finish_position ~ driver_quality_z + C(race_id)", data=df
    ).fit()

    m_quality_q = smf.ols(
        "finish_position ~ qualifying_position + driver_quality_z + C(race_id)",
        data=df,
    ).fit()

    r2_quality = m_quality.rsquared
    r2_quality_q = m_quality_q.rsquared

    incremental_r2 = r2_quality_q - r2_quality

    partial_r2 = (
        (r2_quality_q - r2_quality) / (1 - r2_quality)
        if r2_quality < 1
        else np.nan
    )

    return pd.DataFrame(
        [
            {
                "n": len(df),
                "races": df["race_id"].nunique(),
                "r2_quality": r2_quality,
                "r2_quality_qualifying": r2_quality_q,
                "incremental_r2": incremental_r2,
                "partial_r2_qualifying": partial_r2,
            }
        ]
    )


common_input = analysis_data[
    analysis_data["finish_position"].notna()
    & analysis_data["qualifying_position"].notna()
    & analysis_data["driver_quality_z"].notna()
].copy()

common_r2 = (
    common_input.groupby("series", group_keys=True)
    .apply(lambda g: common_sample_r2(g))
    .reset_index(level=0)
    .reset_index(drop=True)
)

print(common_r2)

common_r2_formatted = common_r2.copy()
for col in ["r2_quality", "r2_quality_qualifying", "incremental_r2", "partial_r2_qualifying"]:
    common_r2_formatted[col] = common_r2_formatted[col].round(4)

print(common_r2_formatted)
# Display it in data tab

    series     n  races  r2_quality  r2_quality_qualifying  incremental_r2  \
0       F1  2778    142    0.338750               0.451317        0.112567   
1  IndyCar  2908    113    0.218251               0.272109        0.053858   
2   NASCAR  9695    272    0.225537               0.257979        0.032442   

   partial_r2_qualifying  
0               0.170234  
1               0.068894  
2               0.041890  
    series     n  races  r2_quality  r2_quality_qualifying  incremental_r2  \
0       F1  2778    142      0.3387                 0.4513          0.1126   
1  IndyCar  2908    113      0.2183                 0.2721          0.0539   
2   NASCAR  9695    272      0.2255                 0.2580          0.0324   

   partial_r2_qualifying  
0                 0.1702  
1                 0.0689  
2                 0.0419  


## Comparison to model on lap 1 position

In [67]:
def summarize_race_group(g: pd.DataFrame):
    g = g.sort_values("lap")

    # Qualifying position
    q = g["qualifying_position"].dropna()
    qualifying_position = q.iloc[0] if len(q) > 0 else np.nan

    # Finishing position
    f = g["finish_position"].dropna()
    if len(f) > 0:
        finish_position = f.iloc[0]
    else:
        finish_position = g.loc[g["lap"].idxmax(), "position"]

    # Position on Lap 1
    lap1 = g.loc[g["lap"] == 1, "position"].dropna()
    lap1_position = lap1.iloc[0] if len(lap1) > 0 else np.nan

    # Keep your existing fallback/start position as well
    starting_observed_position = g.loc[g["lap"].idxmin(), "position"]

    laps_completed = g["lap"].max()

    return pd.Series(
        {
            "qualifying_position": qualifying_position,
            "lap1_position": lap1_position,
            "finish_position": finish_position,
            "starting_observed_position": starting_observed_position,
            "laps_completed": laps_completed,
        }
    )

In [68]:
# ============================================================
# CORRELATION: QUALIFYING POSITION VS LAP 1 POSITION
# ============================================================

correlation_summary = []

for series_name, sub in race_results.groupby("series"):

    sub = sub[
        sub["qualifying_position"].notna()
        & sub["lap1_position"].notna()
    ].copy()

    pearson_r, pearson_p = stats.pearsonr(
        sub["qualifying_position"],
        sub["lap1_position"]
    )

    spearman_rho, spearman_p = stats.spearmanr(
        sub["qualifying_position"],
        sub["lap1_position"]
    )

    correlation_summary.append({
        "series": series_name,
        "n": len(sub),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p
    })

correlation_summary = pd.DataFrame(correlation_summary)

print("\nQualifying position vs Lap 1 position:")
print(correlation_summary.round(4))


Qualifying position vs Lap 1 position:
    series     n  pearson_r  pearson_p  spearman_rho  spearman_p
0       F1  2715     0.8084        0.0        0.8131         0.0
1  IndyCar  2841     0.9157        0.0        0.9168         0.0
2   NASCAR  9642     0.9192        0.0        0.9225         0.0


In [70]:
def calculate_lap1_r2(df: pd.DataFrame):

    df = df[
        df["finish_position"].notna()
        & df["lap1_position"].notna()
        & df["driver_quality_z"].notna()
    ].copy()

    # Reduced model:
    # Finish ~ driver quality + race fixed effects
    reduced = smf.ols(
        "finish_position ~ driver_quality_z + C(race_id)",
        data=df
    ).fit()

    # Full model:
    # Finish ~ Lap 1 position + driver quality + race fixed effects
    full = smf.ols(
        "finish_position ~ lap1_position + driver_quality_z + C(race_id)",
        data=df
    ).fit()

    r2_reduced = reduced.rsquared
    r2_full = full.rsquared

    # Incremental R² from Lap 1 position
    incremental_r2 = r2_full - r2_reduced

    # Partial R² for Lap 1 position
    partial_r2 = incremental_r2 / (1 - r2_reduced)

    return pd.Series({
        "n": len(df),
        "races": df["race_id"].nunique(),
        "r2_quality": r2_reduced,
        "r2_quality_lap1": r2_full,
        "incremental_r2_lap1": incremental_r2,
        "partial_r2_lap1": partial_r2
    })


lap1_summary = (
    analysis_data
    .groupby("series")
    .apply(calculate_lap1_r2)
    .reset_index()
)

print("\nLap 1 position partial R²:")
print(lap1_summary.round(4))


Lap 1 position partial R²:
    series       n  races  r2_quality  r2_quality_lap1  incremental_r2_lap1  \
0       F1  2714.0  141.0      0.3488           0.5158               0.1670   
1  IndyCar  2833.0  113.0      0.2223           0.2896               0.0673   
2   NASCAR  9619.0  272.0      0.2242           0.2615               0.0373   

   partial_r2_lap1  
0           0.2565  
1           0.0865  
2           0.0480  
